In [ ]:
# Note do not commit this file to git with any outputs. Clear all outputs before committing.
# 
# Necessary packages to import for this notebook
# Standard
import json
# Installed
import numpy as np
from matplotlib import pyplot as plt
# Local
from libera_rad.radiometer.radiance import create_emitted_power_interpolation
from libera_rad.calibration.calibration_models import LiberaGroundCalibration
from libera_rad.calibration.constants import ChannelName, BoardName, HousekeepingTemperatureCoefficient
from libera_rad.radiometer.radiance import (calculate_thermistor_power, calculate_heater_max_power)

# Libera L1B Radiometer Science Algorithm
## Introduction
This notebook contains the Libera L1B Radiometer science algorithm. The algorithm is used to process raw electronics data from a radiometer and convert it into a calibrated, filtered radiance and follows the procedure in the associated Algorithm Theoretical Basis Document (ATBD) that will be discussed and elaborated on here. 

### Algorithm Description
At its root, this algorithm has four phases
1. Calculate a nanowatts per dn value from calibration and on orbit information used for converting the raw electronics data to radiance.
2. Calculate the radiance from the radiometer using the pulse width modulation (PWM) signal and ground calibration geometry information.
3. Apply a signal processing filter to match the point response to the existing CERES instruments.
4. Generate the geolocation information to match the radiometer data.

This notebook contains phases 1 and 2 of the algorithm. The other phases will be given in separate notebooks.

### Algorithm Inputs/Outputs
#### Inputs
1. Raw Electronics Data at 200 Hz
2. Ground Calibration Detector Information
3. Ground Calibration Solid Angle Information
4. Ground Calibration Transient Gain Function
5. On-orbit Calibration Transient Gain Deviation Function

#### Outputs
1. Calibrated, filtered Radiance Data at 100 Hz

# Phase 1: Creating a nanowatts per dn value
Realistic Assumptions for this example:
1. Use only the shortwave channel for the radiometer
2. Assume the PWM value ranges from 0 to 12,500 dn with a dark state or mean state of 10,000
3. Assume the temperature of the board varies from 34 to 36 degrees Celsius

## Part 1: The physical approach
The power generated by the detectors can be calculated from two independent physical processes and using these two methods together in an iterative manner is how we can calculate the power output as it depends on the interactions of the dn counting process for the pulse width modulation (PWM) signal. The iteration process can be described as:
1. Assume a starting temperature of the detector
2. Calculate the power generated by the detector at the estimated temperature using the electronic process
3. Use this calculated power along with the thermodynamic process to calculate an updated temperature of the detector
4. Repeat steps 2 and 3 until the temperature of the detector converges to a stable value (only needs 3-6 iterations for instrument team approved convergence)
This calculation is captured in the `calculate_physical_nanowatts_per_dn` function in the `libera_rad.radiance` module and this notebook serves to help explain the process.
### Physical Process 1: Thermodynamics
The detector is a thermodynamic system that is in equilibrium with the environment. The power generated by the detector can be seen as the sum of radiative and conductive heat transfer processes. The radiative heat transfer is given by the Stefan-Boltzmann law and the conductive heat transfer is given by the Fourier law. The power generated by the detector can be expressed as:
\begin{equation*}
P_{\text{detector}} = P_{\text{rad}} + P_{\text{cond}}
\end{equation*}
Where:
\begin{equation*}
P_{\text{rad}} = \epsilon_{\text{ground}} \sigma \left( T_{\text{detector}}^4 - T_{\text{bench}}^4 \right)
\end{equation*}

\begin{equation*}
P_{\text{cond}} = k_{\text{ground}} \left(T_{\text{detector}} - T_{\text{bench}}\right)
\end{equation*}
Where:
- $\epsilon_{\text{ground}}$ is the emissivity of the detector measured as part of ground calibration
- $\sigma$ is the Stefan-Boltzmann constant
- $T_{\text{detector}}$ is the temperature of the detector
- $k_{\text{ground}}$ is the thermal conductivity of the detector measured as part of ground calibration
- $T_{\text{bench}}$ is the temperature of the bench
- $P_{\text{rad}}$ is the radiative power
- $P_{\text{cond}}$ is the conductive power
- $P_{\text{detector}}$ is the total power

In [ ]:
# Use ground calibration data to generate a power vs temperature interpolation for the detector from thermodynamic processes

# Load the ground calibration data for the detector
with open("../libera_rad/data/l1b_ground_calibration.json") as f:
    ground_calibration = json.load(f)
ground_calibration_data = LiberaGroundCalibration(**ground_calibration)
# Select only the shortwave channel calibration data
channel_calibration = ground_calibration_data.channels[ChannelName.SHORTWAVE.value]

# Assume the temperature of the surrounding (the heat sink for thermodynamic purposes) is at 35 degrees Celsius
heat_sink_temp = 35

# Temperature range in Kelvin
temp_range = np.arange(heat_sink_temp, heat_sink_temp + 100, 0.1) + 273.15
emitted_power_range = create_emitted_power_interpolation(temp_range, heat_sink_temp, channel_calibration)

# Plot the power vs temperature curve
plt.plot(temp_range-273.15, emitted_power_range)
plt.xlabel("Temperature (C)")
plt.ylabel("Power (W)")

### Physical Process 2: Electronics
The detector is also an electronic system where power can be calculated as the product of the voltage and current through different components. The detector electronics consist of two circuitry paths, the heater and the thermistor, that combine to represent the total power of the system. 
The total power of the detector can be expressed as:
\begin{equation*}
P_{\text{detector}} = P_{\text{heater}} + P_{\text{thermistor}}
\end{equation*}
Where:
\begin{equation*}
P_{\text{heater}} = I_{\text{heater}}^2 R_{\text{heater}}
\end{equation*}

\begin{equation*}
P_{\text{thermistor}} = I_{\text{thermistor}}^2 R_{\text{thermistor}}
\end{equation*}
Where:
- $I_{\text{heater}}$ is the current through the heater
- $R_{\text{heater}}$ is the total resistance of all elements in the heater path
- $I_{\text{thermistor}}$ is the current through the thermistor
- $R_{\text{thermistor}}$ is the resistance of all elements in the thermistor path
- $P_{\text{heater}}$ is the power generated by the heater
- $P_{\text{thermistor}}$ is the power generated by the thermistor
- $P_{\text{detector}}$ is the total power

As shown in the equations above, the power generated by the detector from an electronics sense depends on two pathways. The heater and the thermistor, simplified circuit diagrams of both are shown below separately, while they do existing geometrically next to each other on the back of the detectors
### Heater Circuit
![Heater Circuit](./images/heater_simple_diagram.png)

Where: 
- $V_{\text{Ref}}$ is the reference voltage across the heater
- $R_{\text{TA}}$ is the resistance of the top resistor (on the circuit board not the detector) for the active detector
- $R_{\text{TA'}}$ is the resistance of the top resistor with the addition of the MOSFET 
- $R_{\text{HA}}$ is the resistance of the heater for the active detector including the trace on the circuit board

The left depiction of the circuit is included to show the usage of the MOSFET as the pulse-width modulation of a straightforward circuit. The complexity of calculation comes with the reality of the circuit trace and its geometry on the back side of the detector. 

From this the current through the heater can be calculated as:
\begin{equation*}
I_{\text{heater}} = \frac{V_{\text{Ref}}}{R_{\text{TA'}} + R_{\text{HA}}}
\end{equation*}
and the resistance as:
\begin{equation*}
R_{\text{heater}} = R_{\text{HA}}
\end{equation*}

In [ ]:
# Use the calibrations for the flight board called EMFPE
pcb_board_calibrations = ground_calibration_data.boards[BoardName.EMFPE.value]

# Use a guess of a first temperature to calculate the power generated by the detector from the electronics process
temperature_guess = 55

# Calculate the power generated by the detector from the heater using the guess temperature and the heat sink temperature from the first step above
heater_power = calculate_heater_max_power(temperature_guess, heat_sink_temp, 
                                          channel_specific_calibration_data=channel_calibration, 
                                          board_specific_calibration_data=pcb_board_calibrations)

### Thermistor Circuit
![Thermistor Circuit](./images/thermistor_simple_diagram.png)

Where:
- $V_{\text{AC}}$ is the reference AC voltage across the bridge
- $R_{\text{Active}}$ is the resistance of the thermistor on the active detector with the trace on the circuit board
- $R_{\text{Ref}}$ is the resistance of the thermistor on the reference detector with the trace on the circuit board
- $R_{\text{B1}}$ is the resistance of the bottom resistor located in the path of the active detector 
- $R_{\text{B2}}$ is the resistance of the bottom resistor located in the path of the reference detector

To understand this thermistor circuit you will need the basics of how these detectors are designed and how they work.
### Detector Design
A single channel measurement on Libera requires two bolometers on the same circuit board. One bolometer is called the active detector and is the one that will be exposed to light. The other is referred to as the reference detector and will not be exposed to light. The reference detector is not essential to any of the calculations here, but is essential to the functioning of the closed loop design of these devices. In the diagram above, the left path is the thermistor of the active detector and the right path is the thermistor of the reference detector. For calculations for the rest of this, we will only be using the thermistor of the active detector.
 
This circuit is a bridge circuit that is used to maintain the resistance of the active detector thermistor to that of the thermistor on the reference . The resistance of the thermistor is used by closed loop design of the circuit to control the temperature of the detector by adjusting the current through the heater. The details of this are available in the ATBD and associated documents and are beyond the scope of this notebook.

The current through the thermistor can be calculated as:
\begin{equation}
I_{\text{thermistor}} = \frac{V_{\text{AC}}}{R_{\text{Active}} + R_{\text{B1}}}
\end{equation}
and the resistance as:
\begin{equation}
R_{\text{thermistor}} = R_{\text{Active}}
\end{equation}

The details of determining resistances are temperature dependent and are calculated as part of the ground calibration process. The details of this and the geometric resistance calculations are beyond the scope of this notebook and are included in the ATBD. The implementations of these calculations are done in the `calculate_resistance_from_temp` and the `calculate_geometric_resistance` functions in the `libera_rad.radiance` module and this notebook serves to help explain the process more broadly.

In [ ]:
# Calculate the power generated by the detector from the thermistor using the guess temperature and the heat sink temperature from the first step above
thermistor_power = calculate_thermistor_power(temperature_guess, heat_sink_temp, 
                                              channel_specific_calibration_data=channel_calibration,
                                              board_specific_calibration_data=pcb_board_calibrations)

Now that we can calculate both the power generated by the detector from the thermodynamic and electronic processes, we can calculate the power generated by the detector from the detector by iterating these two processes. This is done in the `calculate_physical_nanowatts_per_dn` function in a single step and is shown here for understanding.

In [ ]:
# Starting temperature guess in Celsius
previous_temp = 0
estimated_temp = 35
iteration = 0

# Estimated mean operating PWM value
mean_duty_cycle = 10000 / 12500

# Iterate until the temperature converges to a 1 ppm level at the nW level (1e-9*1e-6)
while abs(estimated_temp - previous_temp) > 1e-15:
    previous_temp = estimated_temp
    max_heater_power = calculate_heater_max_power(estimated_temp,
                                                  heat_sink_temp,
                                                  channel_specific_calibration_data=channel_calibration,
                                                  board_specific_calibration_data=pcb_board_calibrations)

    thermistor_power = calculate_thermistor_power(estimated_temp,
                                                  heat_sink_temp,
                                                  channel_specific_calibration_data=channel_calibration,
                                                  board_specific_calibration_data=pcb_board_calibrations)

    # Total Power from the circuit perspective
    mean_heater_power = max_heater_power * mean_duty_cycle
    total_power = mean_heater_power + thermistor_power

    # Estimate the temperature using the emitted power calculation from the beginning
    estimated_temp = np.interp(total_power, emitted_power_range, temp_range) - 273.15
    print("Iteration: ", iteration, "Estimated Temperature: ", estimated_temp)
    iteration += 1


In [ ]:
# Calculate the nanowatts per dn value now that the temperature has converged as the max heater power divided by the mean operating PWM value
watts_per_dn = (max_heater_power / 12500)
print("Nanowatts per dn: ", watts_per_dn*1e9)

# Part 2: The numerical approach
After many iterations of working with this data, a simpler method was found using a numerical fit to two quadratic polynomials using only two parameters: the temperature of the bench board (called the heat sink temperature in the physical approach) and the active duty cycle of the PWM when the detector is in a mean state.

The nw per dn equation can be expressed as:
\begin{equation*}
\text{nW per dn} = c_0 + a_1 \cdot T_{\text{board}}^3 + a_2 \cdot T_{\text{board}}^2 + b_1 \cdot \text{DC}_{\text{active}} + b_2 \cdot \text{DC}_{\text{active}}^2 + c_1 \cdot (T_{\text{board}} \cdot \text{DC}_{\text{active}})
\end{equation*}

Where:
- $T_{\text{board}}$ is the difference in temperature on the board compared to a ground calibrated constant temperature
- $\text{DC}_{\text{active}}$ is the mean state active duty cycle of the PWM signal
- $c_0$ is a constant term
- $a_1$ is a linear coefficient for the temperature term
- $a_2$ is a quadratic coefficient for the temperature term
- $b_1$ is a linear coefficient for the duty cycle term
- $b_2$ is a quadratic coefficient for the duty cycle term
- $c_1$ is a cross term between the temperature and duty cycle terms

This method is implemented generally in the `calculate_numerical_nanowatts_per_dn` function in the `libera_rad.radiance` module and the details of that method are shown here with a few assumptions for understanding.

In [ ]:
# Filter down to the relevant calibration data for easy passing through functions
electronics_calibration_info = ground_calibration_data.boards[BoardName.EMFPE.value]
channel_calibration_info = ground_calibration_data.channels[ChannelName.SHORTWAVE.value]
board_temp_coefficients = ground_calibration_data.housekeeping_temperature_coefficients[HousekeepingTemperatureCoefficient.BENCH_COEFFICIENTS.value]

# Extract numerical coefficients used for the radiance calculation
radiance_coefficients = channel_calibration_info.radiance_coefficients

# Calculate a dark PWM level as a baseline
active_dc = 10000 / 12500

# Calculate the relevant temperatures needed for the radiance calculation
bench_temp = 35
reference_temp = radiance_coefficients.t0_per_dn

nw_per_dn = radiance_coefficients.constant_offset + \
            radiance_coefficients.temp_difference_linear * (bench_temp - reference_temp) + \
            radiance_coefficients.temp_difference_quadratic * (bench_temp - reference_temp) ** 2 + \
            radiance_coefficients.mean_state_offset_linear * active_dc + \
            radiance_coefficients.mean_state_offset_quadratic * active_dc ** 2 + \
            radiance_coefficients.mean_state_vs_temperature_crossover * (bench_temp - reference_temp) * active_dc

print("For comparison:")
print("Nanowatts per dn from numerical estimation: ", nw_per_dn)
print("Nanowatts per dn from physical estimation: ", watts_per_dn*1e9)

# Phase 2: Calculating the radiance
In comparison to the previous step, this one is relatively straightforward. The radiance calculation is done using the following equation:
\begin{equation*}
\text{Radiance} = \frac{(\text{PWM}) \cdot (\text{nW per dn})}{\text{Solid Angle} \cdot \text{Area}_{\text{detector}}}
\end{equation*}

Where:
- $\text{PWM}$ is the pulse width modulation signal
- $\text{nW per dn}$ is the nanowatts per dn value calculated in the previous step using either method
- $\text{Solid Angle}$ is the solid angle of the detector calculated as part of the ground calibration
- $\text{Area}_{\text{detector}}$ is the area of the detector calculated as part of the ground calibration
- $\text{Radiance}$ is the radiance of the detector

This calculation is done in full the `calculate_radiance_from_dn` function in the `libera_rad.radiance` module and is shown here for understanding.

In [ ]:
pwm_dn_data = 10009
power_nw = nw_per_dn * pwm_dn_data

# Convert the power to radiance in units of W m^-2 sr^-1
power_watts = power_nw * 1e-9
# Channel calibration is still for the shortwave channel
radiance = power_watts / (channel_calibration_info.collection_area * channel_calibration_info.solid_angle)

print("Radiance: ", radiance)

# Conclusion
This concludes the first two phases of the L1B radiometer algorithm. The next two phases will be given in separate notebooks.